In [113]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

In [114]:
def get_sp500_tickers():
  table = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
  return table[0]['Symbol'].str.replace('.', '-', regex=False).tolist()

In [115]:
def add_moving_averages(df: pd.DataFrame, spans = [5, 10, 20, 50]):
  for span in spans:
    df[f"MA{span}"] = df["Close"].rolling(window=span).mean()
  
  return df

In [116]:
def add_rsi(df: pd.DataFrame):
  delta = df["Close"].diff()

  up = delta.clip(lower=0)
  down = -1 * delta.clip(upper=0)

  avg_gain = up.rolling(window=14).mean()
  avg_loss = down.rolling(window=14).mean()

  rs = avg_gain / avg_loss

  df["RSI14"] = 100 - (100 / (rs + 1))

  return df

In [117]:
def add_standard_deviations(df: pd.DataFrame, spans=[5, 10, 20, 50]):
  for span in spans:
    df[f"SD{span}"] = df["Close"].rolling(window=span).std()

  return df

In [118]:
def add_rate_of_change(df: pd.DataFrame, spans=[5, 10, 20, 50]):
  for span in spans:
    df[f"ROC{span}"] = df["Close"].pct_change(periods=span)

  return df

In [119]:
def add_avg_true_range(df: pd.DataFrame):
  df["HighLow"] = df["High"] - df["Low"]
  df["HighClose"] = (df["High"] - df["Close"].shift()).abs()
  df["LowClose"] = (df["Low"] - df["Close"].shift()).abs()

  df["TR"] = df[[
    "HighLow",
    "HighClose",
    "LowClose"
  ]].max(axis=1)

  df["ATR14"] = df["TR"].rolling(window=14).mean()

  df.drop(
    columns=[
      "HighLow",
      "HighClose",
      "LowClose",
      "TR"
    ],
    inplace=True
  )

  return df

In [120]:
def add_bollinger_bands(df: pd.DataFrame, window=20, no_std=2):
  rolling_mean = df["Close"].rolling(window=window).mean()
  rolling_std = df["Close"].rolling(window=window).std()

  df["BollMid"] = rolling_mean
  df["BollUpper"] = rolling_mean + (no_std * rolling_std)
  df["BollLower"] = rolling_mean - (no_std * rolling_std)

  df["BollBandwidth"] = df["BollUpper"] - df["BollLower"]

  return df

In [121]:
def add_ema(df: pd.DataFrame, spans=[5, 12, 26, 50]):
  for span in spans:
    df[f"EMA{span}"] = df["Close"].ewm(span=span).mean()

  return df

In [122]:
def add_macd(df: pd.DataFrame, short_window=12, long_window=26, signal_window=9):
  short_ema = df["Close"].ewm(span=short_window, adjust=False).mean()
  long_ema = df["Close"].ewm(span=long_window, adjust=False).mean()

  df["MACD"] = short_ema - long_ema
  df["MACD_Signal"] = df["MACD"].ewm(span=signal_window, adjust=False).mean()
  df["MACD_Histo"] = df["MACD"] - df["MACD_Signal"]

  return df

In [123]:
def add_donchian_channels(df: pd.DataFrame, window=20):
    df["DC_Lower"] = df["Low"].rolling(window=window).min()
    df["DC_Upper"] = df["High"].rolling(window=window).max()
    df["DC_Mid"] = (df["High"] + df["Low"]) / 2
    df["DC_Width"] = df["DC_Upper"] - df["DC_Lower"]
    
    df["DC_Breakout_Strength"] = (df["Close"] - df["DC_Mid"]) / df["DC_Width"]
    
    return df

In [124]:
def add_volume_features(df: pd.DataFrame):
  df["Vol_MA10"] = df["Volume"].rolling(window=10).mean()
  df["Vol_MA20"] = df["Volume"].rolling(window=20).mean()
  
  df["Vol_MA10_Ratio"] = df["Volume"] / df["Vol_MA10"]
  df["Vol_MA20_Ratio"] = df["Volume"] / df["Vol_MA20"]
  
  df["Vol_Spike"] = (df["Vol_MA10_Ratio"] > 1.5).astype(int)
  
  vol_mean = df["Volume"].rolling(window=20).mean()
  vol_std = df["Volume"].rolling(window=20).std()
  
  df["Vol_Z"] = (df["Volume"] - vol_mean) / vol_std
  
  return df

In [125]:
def add_mfi(df: pd.DataFrame, period=14):
  typical_price = (df["High"] + df["Low"] + df["Close"]) / 3

  raw_money_flow = typical_price * df["Volume"]

  delta_tp = typical_price.diff()

  pos_flow = raw_money_flow.where(delta_tp > 0, 0.0)
  neg_flow = raw_money_flow.where(delta_tp < 0, 0.0)

  pos_sum = pos_flow.rolling(window=period).sum()
  neg_sum = neg_flow.rolling(window=period).sum()

  money_flow_ratio = pos_sum / (neg_sum + 1e-9) # avoid div by zero

  money_flow_index = 100 - (100 / (1 + money_flow_ratio))

  df["MFI"] = money_flow_index

  return df

In [126]:
def add_z_close(df: pd.DataFrame):
  z_mean = df["Close"].rolling(window=20).mean()
  z_std = df["Close"].rolling(window=20).std()

  df["ZClose20"] = (df["Close"] - z_mean) / z_std

  return df

In [127]:
def add_stochastic_oscillator(df: pd.DataFrame, k_period=14, d_period=3):
  high_max = df["High"].rolling(window=k_period).max()
  low_min = df["Low"].rolling(window=k_period).min()

  df["Stoch_K%"] = 100 * (
    (df["Close"] - low_min)
    /
    (high_max - low_min + 1e-9)
  ) # avoid div by zero
  
  df["Stoch_D%"] = df["Stoch_K%"].rolling(window=d_period).mean()

  return df

In [128]:
def add_target(df: pd.DataFrame):
  df["FutureClose"] = df["Close"].shift(-1)

  df["Target"] = (df["FutureClose"] > df["Close"]).astype(int)

  return df

In [129]:
FEATURES = [
  add_target,
  add_moving_averages,
  add_rsi,
  add_standard_deviations,
  add_rate_of_change,
  add_avg_true_range,
  add_bollinger_bands,
  add_macd,
  add_ema,
  add_donchian_channels,
  add_volume_features,
  add_mfi,
  add_z_close,
  add_stochastic_oscillator
]

In [130]:
def derive_features(df: pd.DataFrame) -> pd.DataFrame:
  for f in FEATURES:
    df = f(df)

  return df

In [131]:
def process_ticker(ticker, interval="1d", period="20y"):
  try:
    df = yf.download(
      ticker,
      interval=interval,
      period=period,
      auto_adjust=True,
      progress=False
    )

    if df.empty or len(df) < 50:
      return None
    
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    df = derive_features(df=df)

    df.dropna(inplace=True)

    df.columns = pd.MultiIndex.from_product([df.columns, [ticker]])
    df.columns.names = ["Feature", "Ticker"]

    return df

  except Exception as e:
    print(f"Error for ticker ${ticker}: {e}")
    return None

In [132]:
def build_dataset():
  tickers = get_sp500_tickers()
  all_dfs = []

  for ticker in tqdm(tickers, desc="Processing S&P500 Tickers"):
    df = process_ticker(ticker)

    if df is not None:
      all_dfs.append(df)

  if not all_dfs:
    raise Exception("No data collected!")
  
  full_df = pd.concat(all_dfs, axis=1)

  return full_df

In [133]:
build_dataset().to_parquet("sp500_1d_tech_indicators.parquet", engine="pyarrow", index=True)
print("✅ Saved dataset to parquet.")

Processing S&P500 Tickers: 100%|██████████| 503/503 [01:52<00:00,  4.45it/s]


✅ Saved dataset to parquet.
